In [1]:
# @title 1. 환경 설정 및 데이터 전처리 (Environment Setup & Data Preprocessing)
!pip install -q torch-geometric

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn.conv import MessagePassing
from torch_geometric.utils import degree, to_undirected
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

# 재현성을 위한 시드 설정
SEED = 2025
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 데이터 로드
# Colab에 'amazon_train.csv'가 업로드되어 있다고 가정합니다.
try:
    df = pd.read_csv('amazon_train.csv')
    df.columns = ['user', 'item', 'rating'] # 컬럼명 통일
except FileNotFoundError:
    print("오류: 'amazon_train.csv' 파일을 업로드해주세요.")
    # 더미 데이터 생성 (테스트용)
    df = pd.DataFrame({
        'user': ['U'+str(i) for i in range(100)] * 5,
        'item': ['I'+str(i) for i in range(200)] * 2 + ['I'+str(i) for i in range(100)],
        'rating': np.random.randint(1, 6, 500)
    })

# 1. ID 매핑 (String -> Integer)
user_mapping = {u: i for i, u in enumerate(df['user'].unique())}
item_mapping = {i: j for j, i in enumerate(df['item'].unique())}
num_users = len(user_mapping)
num_items = len(item_mapping)

df['user_idx'] = df['user'].map(user_mapping)
df['item_idx'] = df['item'].map(item_mapping)

# 사용자별 이력 수 계산 (제약 조건 적용용)
user_history_counts = df.groupby('user')['item'].count().to_dict()

print(f"총 사용자: {num_users}, 총 아이템: {num_items}, 총 상호작용: {len(df)}")

# 2. 학습/테스트 분할
# 과제 규칙: 평점 무시, 구매 여부만 고려. 모든 데이터를 Positive로 간주.
# 하지만 평가를 위해 일부를 Test로 분리 (마지막 아이템 Leave-one-out 등)
# 여기서는 간단히 8:2 랜덤 스플릿 적용 (실제 과제 시 제공된 test 셋이 있다면 그것을 사용)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=SEED)

# 3. 엣지 인덱스 생성 (PyG 포맷)
# LightGCN은 무방향 그래프로 취급하므로 양방향 엣지 생성은 모델 내부 혹은 여기서 처리
train_user = torch.LongTensor(train_df['user_idx'].values)
train_item = torch.LongTensor(train_df['item_idx'].values)

edge_index = torch.stack([train_user, train_item], dim=0).to(device)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.7 MB/s eta 0:00:00
Using device: cuda
총 사용자: 256009, 총 아이템: 74233, 총 상호작용: 568263


In [2]:
# @title 2. LightGCN 모델 구현 (LightGCN Architecture)

class LightGCN(MessagePassing):
    def __init__(self, num_users, num_items, embedding_dim=64, num_layers=3):
        super().__init__(aggr='add') # 이웃 노드 feature 합산
        self.num_users = num_users
        self.num_items = num_items
        self.embedding_dim = embedding_dim
        self.num_layers = num_layers

        # 초기 임베딩 (User + Item)
        # 정규분포 초기화가 LightGCN에서 성능이 좋음 [He et al., 2020]
        self.embedding = nn.Embedding(num_users + num_items, embedding_dim)
        nn.init.normal_(self.embedding.weight, std=0.1)

    def forward(self, edge_index):
        # 1. 그래프 라플라시안 정규화 상수 계산
        # D^{-1/2} A D^{-1/2}
        edge_index_norm = self.get_norm_adj(edge_index)

        # 2. 임베딩 전파 (Propagation)
        emb_0 = self.embedding.weight
        embs = [emb_0]
        emb_k = emb_0

        for i in range(self.num_layers):
            # Message Passing: x_i = Sum(Norm * x_j)
            emb_k = self.propagate(edge_index_norm, x=emb_k)
            embs.append(emb_k)

        # 3. 레이어 결합 (Layer Combination)
        # 단순히 평균(Mean) 또는 가중 합(1/(K+1)) 사용. 여기서는 평균 사용.
        embs = torch.stack(embs, dim=1)
        final_emb = torch.mean(embs, dim=1)

        # User, Item 임베딩 분리
        users_emb, items_emb = torch.split(final_emb, [self.num_users, self.num_items])
        return users_emb, items_emb

    def get_norm_adj(self, edge_index):
        # 사용자-아이템 이분 그래프를 단일 그래프로 변환 (User ID: 0~N-1, Item ID: N~N+M-1)
        # 아이템 인덱스 쉬프트
        users = edge_index  # [수정됨] 0번째 행(User ID들)만 추출
        items = edge_index[1] + self.num_users # [수정됨] 1번째 행(Item ID들)만 추출하여 Shift

        # 무방향 그래프로 변환 (User <-> Item)
        row = torch.cat([users, items])
        col = torch.cat([items, users])
        edge_index_all = torch.stack([row, col], dim=0)

        # 정규화 계수 계산
        deg = degree(col, self.num_users + self.num_items, dtype=torch.float)
        deg_inv_sqrt = deg.pow(-0.5)
        deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]

        return edge_index_all, norm

    def message(self, x_j, norm):
        # 정규화된 이웃 feature 전달
        # norm은 엣지별 가중치 (1/sqrt(deg_u * deg_i))
        return norm.view(-1, 1) * x_j # [E, 1] *

    def propagate(self, edge_index_info, x):
        edge_index, norm = edge_index_info
        return super().propagate(edge_index, x=x, norm=norm)

In [3]:
# @title 3. 학습 설정 (Training Setup: BPR Loss & Optimizer)

from torch_geometric.nn.conv import MessagePassing # MessagePassing 클래스 임포트 추가

# LightGCN 모델 구현 (LightGCN Architecture) - 원본 정의의 오류 수정을 위해 여기에 재정의됩니다.
class LightGCN(MessagePassing):
    def __init__(self, num_users, num_items, embedding_dim=64, num_layers=3):
        super().__init__(aggr='add') # 이웃 노드 feature 합산
        self.num_users = num_users
        self.num_items = num_items
        self.embedding_dim = embedding_dim
        self.num_layers = num_layers

        # 초기 임베딩 (User + Item)
        # 정규분포 초기화가 LightGCN에서 성능이 좋음 [He et al., 2020]
        self.embedding = nn.Embedding(num_users + num_items, embedding_dim)
        nn.init.normal_(self.embedding.weight, std=0.1)

    def forward(self, edge_index):
        # 1. 그래프 라플라시안 정규화 상수 계산
        # D^{-1/2} A D^{-1/2}
        edge_index_norm = self.get_norm_adj(edge_index)

        # 2. 임베딩 전파 (Propagation)
        emb_0 = self.embedding.weight
        embs = [emb_0]
        emb_k = emb_0

        for i in range(self.num_layers):
            # Message Passing: x_i = Sum(Norm * x_j)
            emb_k = self.propagate(edge_index_norm, x=emb_k)
            embs.append(emb_k)

        # 3. 레이어 결합 (Layer Combination)
        # 단순히 평균(Mean) 또는 가중 합(1/(K+1)) 사용. 여기서는 평균 사용.
        embs = torch.stack(embs, dim=1)
        final_emb = torch.mean(embs, dim=1)

        # User, Item 임베딩 분리
        users_emb, items_emb = torch.split(final_emb, [self.num_users, self.num_items])
        return users_emb, items_emb

    def get_norm_adj(self, edge_index):
        # 사용자-아이템 이분 그래프를 단일 그래프로 변환 (User ID: 0~N-1, Item ID: N~N+M-1)
        # edge_index는 [2, E] 형태이며, edge_index[0]는 사용자 인덱스, edge_index[1]는 아이템 인덱스입니다.
        users_idx = edge_index[0] # Correctly extract user indices as a 1D tensor
        items_idx = edge_index[1] # Correctly extract item indices as a 1D tensor

        # 아이템 인덱스에 사용자 수를 더해 사용자 ID 공간과 분리합니다.
        items_shifted_idx = items_idx + self.num_users

        # 무방향 그래프 엣지 생성: 사용자 -> 아이템 및 아이템 -> 사용자
        # 사용자 -> 아이템 엣지
        row_u_i = users_idx
        col_u_i = items_shifted_idx

        # 아이템 -> 사용자 엣지
        row_i_u = items_shifted_idx
        col_i_u = users_idx

        # 모든 엣지를 하나로 합쳐 무방향 그래프의 엣지 리스트를 만듭니다.
        row = torch.cat([row_u_i, row_i_u])
        col = torch.cat([col_u_i, col_i_u])
        edge_index_all = torch.stack([row, col], dim=0) # [2, 2 * num_interactions]

        # 정규화 계수 계산
        # `col`은 이제 1D 텐서이며, 그래프 내 모든 노드의 인덱스를 포함합니다.
        deg = degree(col, num_nodes=self.num_users + self.num_items)
        deg_inv_sqrt = deg.pow(-0.5)
        deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]

        return edge_index_all, norm

    def message(self, x_j, norm):
        # 정규화된 이웃 feature 전달
        # norm은 엣지별 가중치 (1/sqrt(deg_u * deg_i))
        return norm.view(-1, 1) * x_j # [E, 1] *

    def propagate(self, edge_index_info, x):
        edge_index, norm = edge_index_info
        return super().propagate(edge_index, x=x, norm=norm)

def bpr_loss(users_emb, pos_items_emb, neg_items_emb):
    # Positive Score: u * i
    pos_scores = torch.sum(users_emb * pos_items_emb, dim=1)
    # Negative Score: u * j
    neg_scores = torch.sum(users_emb * neg_items_emb, dim=1)

    # Softplus = log(1 + exp(x)) -> -log_sigmoid 와 유사
    loss = torch.mean(F.softplus(neg_scores - pos_scores))
    return loss

def train(model, optimizer, train_loader, edge_index):
    model.train()
    total_loss = 0

    for batch_users, batch_pos_items in train_loader:
        batch_users = batch_users.to(device)
        batch_pos_items = batch_pos_items.to(device)

        # Negative Sampling (Uniform)
        # 심화: 하드 네거티브 샘플링(DNS)을 적용하면 성능 향상 가능
        batch_neg_items = torch.randint(0, num_items, (batch_users.size(0),)).to(device)

        optimizer.zero_grad()

        # 전체 그래프 전파 후 해당 배치 노드만 선택
        final_users, final_items = model(edge_index)

        user_e = final_users[batch_users]
        pos_e = final_items[batch_pos_items]
        neg_e = final_items[batch_neg_items]

        loss = bpr_loss(user_e, pos_e, neg_e)

        # L2 Regularization (Overfitting 방지)
        reg_loss = (1/2) * (user_e.norm(2).pow(2) +
                            pos_e.norm(2).pow(2) +
                            neg_e.norm(2).pow(2)) / float(len(batch_users))

        loss = loss + 1e-4 * reg_loss

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(train_loader)

# 데이터 로더 준비
class UserItemDataset(torch.utils.data.Dataset):
    def __init__(self, user_tensor, item_tensor):
        self.users = user_tensor
        self.items = item_tensor
    def __len__(self): return len(self.users)
    def __getitem__(self, idx): return self.users[idx], self.items[idx]

train_dataset = UserItemDataset(train_user, train_item)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=1024, shuffle=True)

# 모델 초기화 및 학습
model = LightGCN(num_users, num_items, embedding_dim=64, num_layers=3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 30 # 시간 관계상 30 epoch
print("학습 시작...")
for epoch in range(epochs):
    loss = train(model, optimizer, train_loader, edge_index)
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}")


학습 시작...
Epoch 5/30, Loss: 0.0807
Epoch 10/30, Loss: 0.0243
Epoch 15/30, Loss: 0.0116
Epoch 20/30, Loss: 0.0074
Epoch 25/30, Loss: 0.0056
Epoch 30/30, Loss: 0.0048


In [4]:
# @title 4. test1.csv 기반 최종 추천 결과 생성 (O/X Prediction)

def generate_test_predictions(model, test_file_path, user_history_counts, item_popularity, k_ratio=0.2):
    # 1. Test 파일 로드
    try:
        test_df = pd.read_csv(test_file_path)
        # 컬럼 수 확인 및 처리
        if len(test_df.columns) == 3:
            # 3개의 컬럼이 있다면, 첫 2개는 'user', 'item'으로 가정하고 세 번째는 무시
            test_df.columns = ['user', 'item', 'extra_col_to_drop'] # 임시 컬럼명 할당하여 에러 방지
            test_df = test_df[['user', 'item']] # 필요한 'user'와 'item' 컬럼만 선택
        elif len(test_df.columns) == 2:
            test_df.columns = ['user', 'item'] # 2개의 컬럼이 있다면 그대로 할당
        else:
            print(f"오류: '{test_file_path}' 파일의 컬럼 수가 예상과 다릅니다. (현재: {len(test_df.columns)}개)")
            return None
    except FileNotFoundError:
        print(f"오류: '{test_file_path}' 파일을 찾을 수 없습니다.")
        return None

    model.eval()
    with torch.no_grad():
        final_users, final_items = model(edge_index)

    # 인기도 페널티 항 미리 계산 (Adaptive Threshold)
    beta = 0.1
    # item_mapping 순서대로 인기도 텐서 생성
    pop_list = [item_popularity.get(i, 0) for i in range(num_items)]
    item_pop_tensor = torch.tensor(pop_list).to(device)
    pop_penalty = beta * torch.log(1 + item_pop_tensor)

    # 2. 유저별 추천 리스트 캐싱 (중복 연산 방지)
    # test 파일에 등장하는 유저에 대해서만 Top-K 리스트를 생성합니다.
    target_users = test_df['user'].unique()
    user_recommendations = {} # {user_str: set(item_indices)}

    print("유저별 Top-K 추천 리스트 생성 중...")
    for u_str in tqdm(target_users):
        # 학습 데이터에 없는 유저는 추천 불가 (혹은 인기도 기반 추천) -> 여기선 빈 집합 처리
        if u_str not in user_mapping:
            user_recommendations[u_str] = set()
            continue

        u_idx = user_mapping[u_str]

        # === [핵심 제약 조건 적용] ===
        history_cnt = user_history_counts.get(u_str, 0)
        if history_cnt <= 10:
            top_k = 2 # 이력 10개 이하: 무조건 2개 추천
        else:
            top_k = int(history_cnt * k_ratio) # 그 외: 이력의 20%
            if top_k < 1: top_k = 1

        # 점수 계산
        u_emb = final_users[u_idx]
        scores = torch.matmul(final_items, u_emb)

        # Adaptive Threshold (인기도 페널티 적용)
        scores = scores - pop_penalty

        # 학습 데이터에 있는 아이템 마스킹 (이미 산 건 추천 제외)
        # (구현 복잡도를 위해 생략했으나 성능 위해선 추가 권장)

        # Top-K 선정
        _, indices = torch.topk(scores, top_k)
        user_recommendations[u_str] = set(indices.cpu().numpy())

    # 3. test1.csv의 각 행에 대해 O/X 판별
    results = []
    recommends_count = 0

    print("O/X 판별 중...")
    for _, row in test_df.iterrows():
        u_str = row['user']
        i_str = row['item']

        pred = 'X'
        # 유저가 추천 리스트를 가지고 있고, 해당 아이템이 그 리스트에 있다면 'O'
        if u_str in user_recommendations:
            if i_str in item_mapping:
                i_idx = item_mapping[i_str]
                if i_idx in user_recommendations[u_str]:
                    pred = 'O'
                    recommends_count += 1

        results.append([u_str, i_str, pred])

    # 결과 통계 출력
    total_rows = len(results)
    print("\n" + "="*20)
    print(f"Total recommends (O) = {recommends_count}/{total_rows}")
    print(f"Total not recommends (X) = {total_rows - recommends_count}/{total_rows}")
    print("="*20)

    return pd.DataFrame(results, columns=['user', 'item', 'recommend'])

# 아이템 인기도 계산에 필요한 변수들이 정의되어 있지 않을 경우 다시 로드 (커널 재시작 등)
if 'df' not in globals() or \
   'user_mapping' not in globals() or \
   'item_mapping' not in globals() or \
   'num_users' not in globals() or \
   'num_items' not in globals() or \
   'user_history_counts' not in globals():
    print("Warning: 'df' or related variables not found. Attempting to re-load environment setup.")
    try:
        df = pd.read_csv('amazon_train.csv')
        df.columns = ['user', 'item', 'rating'] # 컬럼명 통일
    except FileNotFoundError:
        print("오류: 'amazon_train.csv' 파일을 업로드해주세요. 더미 데이터를 생성합니다.")
        df = pd.DataFrame({
            'user': ['U'+str(i) for i in range(100)] * 5,
            'item': ['I'+str(i) for i in range(200)] * 2 + ['I'+str(i) for i in range(100)],
            'rating': np.random.randint(1, 6, 500)
        })

    # ID 매핑 (String -> Integer)
    user_mapping = {u: i for i, u in enumerate(df['user'].unique())}
    item_mapping = {i: j for j, i in enumerate(df['item'].unique())}
    num_users = len(user_mapping)
    num_items = len(item_mapping)

    df['user_idx'] = df['user'].map(user_mapping)
    df['item_idx'] = df['item'].map(item_mapping)

    # 사용자별 이력 수 계산 (제약 조건 적용용)
    user_history_counts = df.groupby('user')['item'].count().to_dict()

# 아이템 인기도 계산
item_popularity = df['item_idx'].value_counts().to_dict()

# test1.csv 파일이 업로드되어 있어야 합니다.
final_result_df = generate_test_predictions(model, 'test.csv', user_history_counts, item_popularity)

if final_result_df is not None:
    print(final_result_df.head())
    # 결과를 CSV로 저장
    final_result_df.to_csv('submission.csv', index=False)
    print("저장 완료: submission.csv")

유저별 Top-K 추천 리스트 생성 중...


  0%|          | 0/196 [00:00<?, ?it/s]

O/X 판별 중...

Total recommends (O) = 5/200
Total not recommends (X) = 195/200
             user        item recommend
0  A3SGXH7AUHU8GW  B001E4KFG0         X
1  A1D87F6ZCVE5NK  B00813GRG4         X
2   ABXLMWJIXXAIN  B000LQOCH0         X
3   A30IP3E4MDQ36  B008O3G25W         X
4  A3IVDLIXVJCRDQ  B008O3G25W         X
저장 완료: submission.csv


In [4]:
# @title 4. 제약 조건 기반 추론 및 결과 생성 (Inference with Constraints)

def generate_recommendations_with_constraints(model, all_users, user_history_counts, item_popularity, k_ratio=0.2):
    model.eval()
    results = [] # 빈 리스트로 초기화

    with torch.no_grad():
        final_users, final_items = model(edge_index)

    # 인기도 기반 적응형 임계값 계산을 위한 준비 (Adaptive Threshold)
    # Score(u, i) = Dot(u, i) - beta * log(pop_i) 형태로 구현하여
    # 인기 아이템의 점수를 깎는 방식으로 다양성 확보
    beta = 0.1 # 인기도 페널티 계수 (튜닝 필요)
    item_pop_tensor = torch.tensor([item_popularity.get(i, 0) for i in range(num_items)]).to(device)
    pop_penalty = beta * torch.log(1 + item_pop_tensor)

    for u_idx in tqdm(range(num_users)):
        u_str = list(user_mapping.keys())[list(user_mapping.values()).index(u_idx)]
        history_cnt = user_history_counts.get(u_str, 0)

        # 1. 추천 개수 결정 (Constraints Logic)
        if history_cnt <= 10:
            top_k = 2 # 무조건 2개 추천 (Fallback Logic)
        else:
            top_k = int(history_cnt * k_ratio) # 이력의 20%
            if top_k < 1: top_k = 1 # 최소 1개 보장

        # 2. 점수 계산 및 인기도 페널티 적용
        u_emb = final_users[u_idx] #
        scores = torch.matmul(final_items, u_emb) # [num_items]

        # Adaptive Thresholding: 인기 아이템 점수 하향 조정
        scores = scores - pop_penalty

        # 3. 이미 구매한 아이템 제외 (Masking)
        # 실제 구현시 학습 데이터에 있는 아이템은 -inf로 마스킹
        # (여기서는 간략화를 위해 생략하거나, 별도 마스킹 로직 필요)

        # 4. Top-K 추출
        _, indices = torch.topk(scores, top_k)
        recommended_items = indices.cpu().numpy()

        # 5. 결과 포맷팅 (O/X)
        # Test Set에 있는 아이템에 대해 O/X 판별 (평가용)
        # 과제 요구사항: "추천 결과(O/X)를 출력" -> 테스트 셋의 아이템이 추천 리스트에 포함되었는지 확인
        # 만약 임의의 (User, Item) 쌍에 대한 O/X를 묻는다면 아래 로직 수정 필요

        # 여기서는 해당 유저에게 추천된 아이템 리스트 자체를 출력하는 예시
        for item_idx in recommended_items:
            i_str = list(item_mapping.keys())[list(item_mapping.values()).index(item_idx)]
            results.append([u_str, i_str, 'O'])

    return pd.DataFrame(results, columns=['user', 'item', 'recommend'])

# 아이템 인기도 계산
item_popularity = df['item_idx'].value_counts().to_dict()

# 최종 결과 생성
final_df = generate_recommendations_with_constraints(model, range(num_users), user_history_counts, item_popularity)

# 결과 출력 (샘플)
print("\n=== 최종 추천 결과 샘플 (Constraints 적용됨) ===")
print(final_df.head(10))

# 사용자별 추천 개수 검증
print("\n=== 제약 조건 검증 (User: A395BORC6FGVXV, History <= 10 case check) ===")
sample_user = df['user'].iloc[0] # 예시 유저 (첫 번째 유저 선택)
hist_len = user_history_counts[sample_user]
rec_len = len(final_df[final_df['user'] == sample_user])
print(f"User: {sample_user}, History: {hist_len}, Recommended: {rec_len}")
if hist_len <= 10 and rec_len == 2:
    print("PASS: 이력 10개 이하 사용자에게 2개 추천 조건 만족")
elif hist_len > 10:
    print(f"PASS: 이력 {hist_len}개 사용자에게 {rec_len}개 ({rec_len/hist_len*100:.1f}%) 추천")